## creating incremental flag

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text('incremental_flag','0')

In [0]:
incremental_flag=dbutils.widgets.get('incremental_flag')
print(incremental_flag)

## creating dimension for dealer


In [0]:
df_src=spark.sql('''
select *
from parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/carSales`
''')

In [0]:
df_src=spark.sql('''
select distinct(Dealer_ID),DealerName
from parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/carSales`
''')

## Now import only Schema  to check column



In [0]:
df_sink=spark.sql(
    '''SELECT 1 as Dealer_Key,Dealer_ID,DealerName
FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/carSales`
WHERE 1=0'''
)


### creating dim Branch for initial and incremental loading

In [0]:
try:
    table_exists = spark.catalog.tableExists('cars_catalog.gold.dim_dealer')
except Exception:
    table_exists = False

if table_exists:
        #for Incremental Loading
    df_sink=spark.sql(
    '''SELECT Dealer_Key,Dealer_ID,DealerName
FROM cars_catalog.gold.dim_dealer
'''
)
        #for Initial Loading
else:
    df_sink=spark.sql(
    '''SELECT 1 as Dealer_Key,Dealer_ID,DealerName
FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/carSales`
WHERE 1=0'''
)

### Filtering new records and old records 

In [0]:
df_filter = df_src.join(df_sink,df_src.Dealer_ID==df_sink.Dealer_ID,'left').select(df_sink.Dealer_Key,df_src.Dealer_ID,df_src.DealerName)


### bringing old records


In [0]:
#bring old records in which Branch_Key Is Not Null
df_filter_old = df_filter.filter(df_filter.Dealer_Key.isNotNull())

### bringing new records


In [0]:
#bring new records in which Model_Key Is  Null
#and here we do not need Model_key cz we create surrogate key based on the old records
df_filter_new = df_filter.filter(df_filter.Dealer_Key.isNull()).select("Dealer_ID","DealerName")

### Create  Surrogate key 

In [0]:
#it will fetch the max of surrogate key from the existing table
if incremental_flag == '0':
    max_value=1
else:

    max_vaue_df=spark.sql("select max(Dealer_Key) from cars_catalog.gold.dim_dealer")  #it will list with max of Model_key
    max_value=max_vaue_df.first()[0]      #it will list with max of Model_key
    

   


_now create surrogate key column and add max of surrogate key_

In [0]:
df_filter_new = df_filter_new.withColumn('Dealer_Key', lit(max_value) + monotonically_increasing_id())

### creating a final df with (df_filter_old + df_filter_new)

In [0]:
df_final = df_filter_old.unionByName(df_filter_new)

# SCD Type-1 (Upsert)

In [0]:
#incremental run
try:
    table_exists = spark.catalog.tableExists('cars_catalog.gold.dim_dealer')
except Exception:
    table_exists = False

if table_exists:
    delta_tbl = DeltaTable.forPath(spark, 'abfss://gold@adlscarproject.dfs.core.windows.net/dim_dealer')

    delta_tbl.alias('trg').merge(df_final.alias('src'),'trg.Dealer_ID = src.Dealer_ID')\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

#initial run
else:
    df_final.write.format('delta')\
        .mode("overwrite")\
            .option('path','abfss://gold@adlscarproject.dfs.core.windows.net/dim_dealer')\
                .saveAsTable("cars_catalog.gold.dim_dealer")

In [0]:
%sql
select * from cars_catalog.gold.dim_dealer